# Feature Engineering

---

1. Import packages
2. Load data
3. Feature engineering

---

## 1. Import packages

In [ ]:
import pandas as pd

---
## 2. Load data

In [ ]:
df = pd.read_csv('./clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [ ]:
df.head(3)

---

## 3. Feature engineering

### Difference between off-peak prices in December and preceding January

Below is the code created by your colleague to calculate the feature described above. Use this code to re-create this feature and then think about ways to build on this feature to create features with a higher predictive power.

In [ ]:
price_df = pd.read_csv('../data-analysis/price_data.csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

In [ ]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

Now it is time to get creative and to conduct some of your own feature engineering! Have fun with it, explore different ideas and try to create as many as you can!

In [ ]:
# attach collegue's code to merge the difference with the main dataframe

df = df.merge(diff, how='left', on='id')

df.head(5)

In [ ]:
# dec minus jan prices for all six prices 

price_cols = ['price_off_peak_var', 'price_off_peak_fix', 'price_peak_var', 'price_peak_fix', 'price_mid_peak_var', 'price_mid_peak_fix']

price_sorted = price_df.sort_values(by=['id', 'price_date'])

g = price_sorted.groupby('id')[price_cols]

dec_jan = (g.last() - g.first()).add_prefix('diff_dec_jan_').reset_index()

# g.head()

dec_jan

In [ ]:
# mean and std for each price per customer

stats = g.agg(['mean', 'std'])
stats.columns = [f'{col}_{stat}' for col, stat in stats.columns]

stats = stats.reset_index()

stats.head()

In [ ]:
# largest month-to-month rise (off-peak energy and power)

month_to_month = price_sorted.copy()

for col in ['price_off_peak_var', 'price_off_peak_fix']:
    month_to_month[f'{col}_mom'] = month_to_month.groupby('id')[col].diff()

# month_to_month.head()

max_jump = month_to_month.groupby('id')[['price_off_peak_var_mom', 'price_off_peak_fix_mom']].max().rename(columns=lambda c: 'max_jump_' + c.replace('_mom', '')).reset_index()


max_jump

In [ ]:
# join to df and check for missing values

print("Missing values in the final dataset:")
print(df.isnull().sum())

for feats in (dec_jan, stats, max_jump):
    df = df.merge(feats, how='left', on='id')

print("\n\nMissing values in the final dataset after merging new features:")
new_cols = [c for c in df.columns if c.startswith(('diff_dec_jan_', 'max_jump_')) or c.endswith(('_mean', '_std'))]
df[new_cols].isna().sum()